# Insurance Data Analytics with DuckDB
This notebook includes NL Query, SQL, visualization, and business insights.

## 1. Create DuckDB Connection
**NL Query:** Create a connection to DuckDB

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect(database=':memory:')

## 2. Load Data
**NL Query:** Load insurance.csv into DuckDB

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE insurance AS
SELECT *
FROM read_csv_auto('insurance.csv');
""")

con.execute("SELECT COUNT(*) FROM insurance").df()

## 3. Top 3 Charges per Region
**Business Insight:** Identify high-cost customers per region

In [ ]:
query = """
SELECT *
FROM (
    SELECT region, age, charges,
           RANK() OVER (PARTITION BY region ORDER BY charges DESC) AS rnk
    FROM insurance
) t
WHERE rnk <= 3;
"""

df = con.execute(query).df()
df

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(range(len(df)), df['charges'])
plt.title("Top 3 Charges per Region")
plt.show()

## 4. Compare to Regional Average

In [ ]:
query = """
SELECT region, age, charges,
       AVG(charges) OVER (PARTITION BY region) AS regional_avg
FROM insurance;
"""

df = con.execute(query).df()

In [ ]:
plt.scatter(df['regional_avg'], df['charges'])
plt.title("Customer vs Regional Avg")
plt.show()

## 5. Above Average Customers

In [ ]:
query = """
SELECT *
FROM (
    SELECT *, AVG(charges) OVER (PARTITION BY region) AS regional_avg
    FROM insurance
) t
WHERE charges > regional_avg;
"""

df = con.execute(query).df()

In [ ]:
plt.hist(df['charges'], bins=30)
plt.title("Above Average Charges")
plt.show()

## 6. % Contribution

In [ ]:
query = """
SELECT charges,
       charges * 100.0 / SUM(charges) OVER () AS pct_total
FROM insurance;
"""

df = con.execute(query).df()

In [ ]:
plt.hist(df['pct_total'], bins=30)
plt.title("% Contribution")
plt.show()

## 7. Running Total by Age

In [ ]:
query = """
SELECT age,
       SUM(charges) AS total_charges,
       SUM(SUM(charges)) OVER (ORDER BY age) AS running_total
FROM insurance
GROUP BY age;
"""

df = con.execute(query).df()

In [ ]:
plt.plot(df['age'], df['running_total'])
plt.title("Running Total by Age")
plt.show()

## 8. Do Smokers Pay More?

In [ ]:
query = """
SELECT smoker, AVG(charges) AS avg_charges
FROM insurance
GROUP BY smoker;
"""

df = con.execute(query).df()

In [ ]:
plt.bar(df['smoker'], df['avg_charges'])
plt.title("Smoker vs Charges")
plt.show()

## 9. BMI vs Charges

In [ ]:
query = "SELECT bmi, charges FROM insurance;"
df = con.execute(query).df()

In [ ]:
plt.scatter(df['bmi'], df['charges'])
plt.title("BMI vs Charges")
plt.show()

## 10. Smokers vs BMI

In [ ]:
query = "SELECT smoker, bmi FROM insurance;"
df = con.execute(query).df()

In [ ]:
for s in df['smoker'].unique():
    subset = df[df['smoker']==s]
    plt.hist(subset['bmi'], alpha=0.5, label=s)

plt.legend()
plt.title("BMI Distribution by Smoker")
plt.show()